In [1]:
from pyspark.sql import SparkSession
import spark

In [2]:
spark = SparkSession.builder \
.appName("DataFrame Basics") \
.getOrCreate()

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/10 15:54:41 INFO SparkEnv: Registering MapOutputTracker
26/04/10 15:54:42 INFO SparkEnv: Registering BlockManagerMaster
26/04/10 15:54:42 INFO SparkEnv: Registering BlockManagerMasterHeartbeat
26/04/10 15:54:42 INFO SparkEnv: Registering OutputCommitCoordinator


### Creating a Dataframe from the data

In [3]:
data = [
(0, "Customer_0", "Bangalore", "Karnataka", "India", "2023-11-11", True),
(1, "Customer_1", "Delhi", "Delhi", "India", "2023-08-26", True)]
columns = ["customer_id", "name", "city", "state", "country", "registration_date","is_active"]

In [4]:
df = spark.createDataFrame(data,columns)

In [5]:
df.show()

+-----------+----------+---------+---------+-------+-----------------+---------+
|customer_id|      name|     city|    state|country|registration_date|is_active|
+-----------+----------+---------+---------+-------+-----------------+---------+
|          0|Customer_0|Bangalore|Karnataka|  India|       2023-11-11|     true|
|          1|Customer_1|    Delhi|    Delhi|  India|       2023-08-26|     true|
+-----------+----------+---------+---------+-------+-----------------+---------+



In [6]:
# Reading local data
df.select('name').show()

+----------+
|      name|
+----------+
|Customer_0|
|Customer_1|
+----------+



### Dataframe - Reading Data from CSV on HDFS

In [7]:
!hadoop fs -ls /tmp/

Found 7 items
-rw-r--r--   2 sowapipython hadoop   10528211 2026-04-01 18:47 /tmp/customers_10mb.csv
-rw-r--r--   2 sowapipython hadoop  168541068 2026-04-01 18:48 /tmp/customers_150mb.csv
-rw-r--r--   2 sowapipython hadoop    1060750 2026-04-01 18:47 /tmp/customers_1mb.csv
-rw-r--r--   2 sowapipython hadoop  343317147 2026-04-01 18:48 /tmp/customers_300mb.csv
drwxrwxrwt   - hdfs         hadoop          0 2026-03-31 14:46 /tmp/hadoop-yarn
drwx-wx-wx   - hive         hadoop          0 2026-03-31 14:47 /tmp/hive
-rw-r--r--   2 root         hadoop         84 2026-04-01 17:13 /tmp/inputhdfsdbz.txt


In [8]:
### I have uploaded the first_100_customers.csv file into the local and kept it in the hadoop files 
### And reading the file from HDFS and also changed the name to customers_100.csv
!hadoop fs -ls /data/

Found 1 items
-rw-r--r--   2 root hadoop       5488 2026-04-10 15:25 /data/customers_100.csv


### Read the file from HDFS

In [9]:
df_2 = spark.read \
.format("csv") \
.option("header", "true") \
.option("inferSchema", "true") \
.load("/data/customers_100.csv")

In [10]:
df_2.show(5)

+-----------+----------+---------+-----------+-------+-----------------+---------+
|customer_id|      name|     city|      state|country|registration_date|is_active|
+-----------+----------+---------+-----------+-------+-----------------+---------+
|          0|Customer_0|     Pune|Maharashtra|  India|       2023-06-29|    false|
|          1|Customer_1|Bangalore| Tamil Nadu|  India|       2023-12-07|     true|
|          2|Customer_2|Hyderabad|    Gujarat|  India|       2023-10-27|     true|
|          3|Customer_3|Bangalore|  Karnataka|  India|       2023-10-17|    false|
|          4|Customer_4|Ahmedabad|  Karnataka|  India|       2023-03-14|    false|
+-----------+----------+---------+-----------+-------+-----------------+---------+
only showing top 5 rows



In [11]:
df_2.printSchema()

root
 |-- customer_id: integer (nullable = true)
 |-- name: string (nullable = true)
 |-- city: string (nullable = true)
 |-- state: string (nullable = true)
 |-- country: string (nullable = true)
 |-- registration_date: date (nullable = true)
 |-- is_active: boolean (nullable = true)



In [13]:
active_customers = df_2.filter('is_active = true')

In [14]:
active_customers.show()

+-----------+-----------+---------+-----------+-------+-----------------+---------+
|customer_id|       name|     city|      state|country|registration_date|is_active|
+-----------+-----------+---------+-----------+-------+-----------------+---------+
|          1| Customer_1|Bangalore| Tamil Nadu|  India|       2023-12-07|     true|
|          2| Customer_2|Hyderabad|    Gujarat|  India|       2023-10-27|     true|
|          7| Customer_7|Ahmedabad|West Bengal|  India|       2023-12-28|     true|
|          8| Customer_8|     Pune|  Karnataka|  India|       2023-06-22|     true|
|          9| Customer_9|   Mumbai|  Telangana|  India|       2023-01-05|     true|
|         10|Customer_10|     Pune|    Gujarat|  India|       2023-08-05|     true|
|         13|Customer_13|  Chennai|  Karnataka|  India|       2023-11-06|     true|
|         15|Customer_15|   Mumbai|    Gujarat|  India|       2023-03-02|     true|
|         18|Customer_18|     Pune|      Delhi|  India|       2023-10-04|   

In [15]:
df_2.select('customer_id').show()

+-----------+
|customer_id|
+-----------+
|          0|
|          1|
|          2|
|          3|
|          4|
|          5|
|          6|
|          7|
|          8|
|          9|
|         10|
|         11|
|         12|
|         13|
|         14|
|         15|
|         16|
|         17|
|         18|
|         19|
+-----------+
only showing top 20 rows



In [17]:
selected_columns = df_2.select('customer_id', 'name', 'city')

In [18]:
selected_columns

DataFrame[customer_id: int, name: string, city: string]

In [19]:
selected_columns.show(6)

+-----------+----------+---------+
|customer_id|      name|     city|
+-----------+----------+---------+
|          0|Customer_0|     Pune|
|          1|Customer_1|Bangalore|
|          2|Customer_2|Hyderabad|
|          3|Customer_3|Bangalore|
|          4|Customer_4|Ahmedabad|
|          5|Customer_5|Hyderabad|
+-----------+----------+---------+
only showing top 6 rows



In [20]:
spark.stop()